# ReasonCritic-7B: Verification/Reasoning Model

**Model:** Qwen2.5-7B-Instruct | **VRAM:** ~10GB | **Time:** ~10h (3 stages)

In [ ]:
# C1: Install + GPU
import subprocess,sys,torch
from pathlib import Path
subprocess.check_call([sys.executable,'-m','pip','install','--no-cache-dir','unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git'])
subprocess.check_call([sys.executable,'-m','pip','install','--no-deps','trl>=0.12.0'])
extras=['transformers>=4.46.0','datasets>=3.0.0','accelerate>=1.1.0','peft>=0.13.0','bitsandbytes>=0.44.0','torch>=2.1.0','huggingface_hub>=0.24.0','tqdm','sentencepiece']
subprocess.check_call([sys.executable,'-m','pip','install','--no-cache-dir',*extras])
import unsloth; print(f'Unsloth {unsloth.__version__}')
import trl; print(f'TRL {trl.__version__}')
if not torch.cuda.is_available(): raise RuntimeError('No GPU!')
print(f'GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB')
OUTPUT_DIR=Path('/content/reason-critic-7b'); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
HF_USERNAME='your-username'; HF_REPO_ID=f'{HF_USERNAME}/ReasonCritic-7B'
def print_vram(stage=''):
    a=torch.cuda.memory_allocated(0)/1024**3;r=torch.cuda.memory_reserved(0)/1024**3;f=(torch.cuda.get_device_properties(0).total_mem/1024**3)-r
    print(f'VRAM{(" ("+stage+")" if stage else "")}: {a:.2f} GB alloc, {r:.2f} GB reserved, {f:.2f} GB free')


In [ ]:
# C2: Load Qwen2.5-7B
from unsloth import FastLanguageModel
import gc
torch.cuda.empty_cache();gc.collect()
MAX_SEQ_LENGTH=4096
model,tokenizer=FastLanguageModel.from_pretrained(model_name='unsloth/Qwen2.5-7B-Instruct',max_seq_length=MAX_SEQ_LENGTH,load_in_4bit=True,dtype=None,trust_remote_code=True)
model=FastLanguageModel.get_peft_model(model,r=32,lora_alpha=64,lora_dropout=0,bias='none',use_gradient_checkpointing='unsloth',random_state=42,use_rslora=False,loftq_config=None,target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} / {sum(p.numel() for p in model.parameters()):,}')
print_vram('after model load')


In [ ]:
# C3: Prepare verification data
from datasets import Dataset
from tqdm.notebook import tqdm
import random
random.seed(42)
SYSTEM_VERIFY='You are ReasonCritic. Analyze code and end with ACCEPT or REJECT.'
VERIFY=[('Sort list','def sort_by_key(items,key):\n    return sorted(items,key=lambda x:x[key])','Correct. Uses sorted() returning new list.\nVerdict: ACCEPT',True),('Read file','def read_file(path):\n    f=open(path)\n    data=f.read()\n    return data','Critical: file handle never closed. Use context manager.\nVerdict: REJECT - resource leak',False),('Palindrome','def is_palindrome(s):\n    return s==s[::-1]','Clean and Pythonic. Efficient slicing.\nVerdict: ACCEPT',True),('Remove dups','def rem(items):\n    for i in items:\n        while items.count(i)>1: items.remove(i)\n    return items','O(n^2) and mutation during iteration. Use dict.fromkeys().\nVerdict: REJECT',False)]
sft=[];dpo=[]
for desc,code,crit,good in VERIFY:
    for _ in range(1250):
        sft.append({'conversations':[{'role':'system','content':SYSTEM_VERIFY},{'role':'user','content':f'Review:\n```python\n{code}\n```\nTask: {desc}'},{'role':'assistant','content':crit}]})
    bad='Code looks fine.' if not good else 'Minor issues.'
    dpo.append({'prompt':f'<|im_start|>system\n{SYSTEM_VERIFY}<|im_end|>\n<|im_start|>user\nReview: {desc}<|im_end|>','chosen':f'<|im_start|>assistant\n{crit}<|im_end|>','rejected':f'<|im_start|>assistant\n{bad}<|im_end|>'})
contrastive_ds=Dataset.from_list(sft);dpo_ds=Dataset.from_list(dpo)
def fmt_sft(ex): return {'text':tokenizer.apply_chat_template(ex['conversations'],tokenize=False,add_generation_prompt=False)}
con_fmt=contrastive_ds.map(fmt_sft)
dpo_fmt=dpo_ds.map(lambda x:{'prompt':x['prompt'],'chosen':x['chosen'],'rejected':x['rejected']})
print(f'Contrastive: {len(con_fmt):,}, DPO: {len(dpo_fmt):,}')
print_vram('data ready')


In [ ]:
# C4: Stage 1 - Contrastive Learning (~4h)
from trl import SFTTrainer
from transformers import TrainingArguments
torch.cuda.empty_cache();gc.collect()
print_vram('before S1')
s1=TrainingArguments(output_dir=str(OUTPUT_DIR/'stage1'),learning_rate=2e-4,num_train_epochs=1,per_device_train_batch_size=2,gradient_accumulation_steps=8,warmup_steps=25,logging_steps=10,save_steps=100,save_total_limit=3,optim='adamw_8bit',weight_decay=0.01,lr_scheduler_type='cosine',seed=42,fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),report_to='tensorboard')
trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=con_fmt,dataset_text_field='text',max_seq_length=MAX_SEQ_LENGTH,args=s1)
try: r=trainer.train(); print(f'S1 done! Loss: {r.training_loss:.4f}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower(): torch.cuda.empty_cache();gc.collect();trainer.args.per_device_train_batch_size=1;trainer.args.gradient_accumulation_steps=16;r=trainer.train();print(f'S1 (recovery)! Loss: {r.training_loss:.4f}')
    else: raise
model.save_pretrained(str(OUTPUT_DIR/'stage1'));tokenizer.save_pretrained(str(OUTPUT_DIR/'stage1'))
print_vram('after S1')


In [ ]:
# C5: Stage 2 - Predictions (~3h)
torch.cuda.empty_cache();gc.collect()
s2=TrainingArguments(output_dir=str(OUTPUT_DIR/'stage2'),learning_rate=1e-4,num_train_epochs=1,per_device_train_batch_size=2,gradient_accumulation_steps=8,warmup_steps=25,logging_steps=10,save_steps=100,save_total_limit=3,optim='adamw_8bit',weight_decay=0.01,lr_scheduler_type='cosine',seed=42,fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),report_to='tensorboard')
trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=con_fmt,dataset_text_field='text',max_seq_length=MAX_SEQ_LENGTH,args=s2)
try: r=trainer.train(); print(f'S2 done! Loss: {r.training_loss:.4f}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower(): torch.cuda.empty_cache();gc.collect();trainer.args.per_device_train_batch_size=1;trainer.args.gradient_accumulation_steps=16;r=trainer.train();print(f'S2 (recovery)! Loss: {r.training_loss:.4f}')
    else: raise
model.save_pretrained(str(OUTPUT_DIR/'stage2'));tokenizer.save_pretrained(str(OUTPUT_DIR/'stage2'))
print_vram('after S2')


In [ ]:
# C6: Stage 3 - DPO (~3h)
from trl import DPOConfig,DPOTrainer
torch.cuda.empty_cache();gc.collect()
dpo_cfg=DPOConfig(output_dir=str(OUTPUT_DIR/'stage3_dpo'),learning_rate=5e-5,num_train_epochs=1,per_device_train_batch_size=1,gradient_accumulation_steps=16,warmup_steps=15,logging_steps=10,save_steps=100,save_total_limit=3,optim='adamw_8bit',weight_decay=0.01,lr_scheduler_type='cosine',seed=42,fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),max_length=MAX_SEQ_LENGTH,max_prompt_length=2048,beta=0.1,loss_type='sigmoid',report_to='tensorboard')
dpo_trainer=DPOTrainer(model=model,ref_model=None,args=dpo_cfg,train_dataset=dpo_fmt,processing_class=tokenizer)
try: r=dpo_trainer.train(); print(f'S3 done! Loss: {r.training_loss:.4f}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower(): torch.cuda.empty_cache();gc.collect();dpo_trainer.args.gradient_accumulation_steps=32;dpo_trainer.args.max_length=2048;r=dpo_trainer.train();print(f'S3 (recovery)! Loss: {r.training_loss:.4f}')
    else: raise
model.save_pretrained(str(OUTPUT_DIR/'stage3_dpo'));tokenizer.save_pretrained(str(OUTPUT_DIR/'stage3_dpo'))
print('All 3 stages complete!')
print_vram('after S3')


In [ ]:
# C7: Evaluate
FastLanguageModel.for_inference(model)
tasks=[{'name':'Correct code','prompt':'Review: def f(n):\n    if n<0: raise ValueError()\n    r=1\n    for i in range(2,n+1): r*=i\n    return r'},{'name':'Bug code','prompt':'Review: def f(p):\n    f=open(p)\n    d=f.read()\n    return d'}]
for t in tasks:
    msgs=[{'role':'system','content':'You are ReasonCritic. End with ACCEPT or REJECT.'},{'role':'user','content':t['prompt']}]
    inp=tokenizer.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')
    with torch.no_grad(): out=model.generate(input_ids=inp,max_new_tokens=256,temperature=0.3,top_p=0.9,do_sample=True)
    resp=tokenizer.decode(out[0][inp.shape[1]:],skip_special_tokens=True)
    print(f"{t['name']}: {resp[:200]}\n")


In [ ]:
# C8: Export GGUF
import shutil
EXPORT_DIR=OUTPUT_DIR/'exports'; EXPORT_DIR.mkdir(parents=True,exist_ok=True)
merged_dir=str(EXPORT_DIR/'reason-critic-7b-16bit')
model.save_pretrained_merged(merged_dir,tokenizer,save_method='merged_16bit')
print(f'16-bit: {merged_dir}')
gguf_dir=str(EXPORT_DIR/'reason-critic-7b-gguf')
try: model.save_pretrained_gguf(gguf_dir,tokenizer,quantization_method='q4_k_m')
except: model.save_pretrained_gguf(gguf_dir,tokenizer,quantization_method='q8_0')
lora_dir=str(EXPORT_DIR/'reason-critic-7b-lora');model.save_pretrained(lora_dir);tokenizer.save_pretrained(lora_dir)
print(f'LoRA: {lora_dir}')


In [ ]:
# C9: Upload
from huggingface_hub import HfApi,create_repo
import os
try: from google.colab import userdata; HF_TOKEN=userdata.get('HF_TOKEN')
except: HF_TOKEN=os.environ.get('HF_TOKEN',None) or input('HF token: ')
if HF_TOKEN:
    api=HfApi(token=HF_TOKEN); create_repo(repo_id=HF_REPO_ID,exist_ok=True,token=HF_TOKEN)
    api.upload_folder(folder_path=merged_dir,repo_id=HF_REPO_ID,token=HF_TOKEN,commit_message='ReasonCritic-7B 16-bit')
    lr=f'{HF_REPO_ID}-LoRA'; create_repo(repo_id=lr,exist_ok=True,token=HF_TOKEN)
    api.upload_folder(folder_path=lora_dir,repo_id=lr,token=HF_TOKEN,commit_message='ReasonCritic-7B LoRA')
    if os.path.exists(gguf_dir):
        gr=f'{HF_REPO_ID}-GGUF'; create_repo(repo_id=gr,exist_ok=True,token=HF_TOKEN)
        for gf in Path(gguf_dir).glob('*.gguf'): api.upload_file(path_or_fileobj=str(gf),path_in_repo=gf.name,repo_id=gr,token=HF_TOKEN)
    print(f'https://huggingface.co/{HF_REPO_ID}')
else: print('No HF token')
print('ReasonCritic-7B Complete!')
